# TSMC SEC 6-K 全64期接受时钟核对
保留 SGML 头与 SEC submissions API 文本，采用较晚日期后的下一实际 Qlib 交易日；不读取市场收益或拟合。

In [1]:
import collections,datetime,hashlib,json,os
from pathlib import Path
import pandas as pd,qlib
from qlib.data import D
root=Path.cwd().resolve();card=root/'research/stage1_tsmc_6k_full_acceptance_clock_audit_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
repair=json.loads((card/'mechanical_nas_field_repair.json').read_text());assert repair['frozen_before_repair_execution'] and repair['plan_sha256']==sha(card/'plan.json')
assert plan['frozen'] and sha(root/src['quarterly_inventory_result'])==src['quarterly_inventory_result_sha256'] and sha(root/src['Qlib_provider']/'qlib_store_manifest.json')==src['Qlib_provider_manifest_sha256']
inventory=json.loads((root/src['quarterly_inventory_result']).read_text());assert inventory['status']=='SEC_TSMC_quarterly_guidance_2011_2026_all_64_months_identified' and len(inventory['matches'])==inventory['matched_months']==src['expected_source_months']==64 and not inventory['unknown_months'] and inventory['NAS_remote_and_scoped_restore_verified']
assert len({x['reporting_month'] for x in inventory['matches']})==64 and all(len(x['accepted_header'])==14 and x['accepted_api'].endswith('Z') for x in inventory['matches'])
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-01',end_time='2026-10-09',freq='day')).normalize();assert calendar.is_monotonic_increasing and calendar.is_unique and calendar[0]<=pd.Timestamp('2011-01-27') and calendar[-1]>=pd.Timestamp('2026-10-09')
rows=[]
for item in inventory['matches']:
 header=pd.Timestamp(datetime.datetime.strptime(item['accepted_header'],'%Y%m%d%H%M%S'))
 api=pd.Timestamp(item['accepted_api']);assert api.tzinfo is not None
 local=header.tz_localize('America/New_York').tz_convert('UTC');fixed=header.tz_localize(datetime.timezone(datetime.timedelta(hours=-5))).tz_convert('UTC')
 api=api.tz_convert('UTC');later_date=max(header.normalize(),api.tz_localize(None).normalize())
 available=calendar[calendar>later_date]
 row={'reporting_month':item['reporting_month'],'accession':item['accession'],'SGML_header':item['accepted_header'],'API_acceptance_text':item['accepted_api'],'SGML_calendar_date':header.strftime('%Y-%m-%d'),'API_literal_UTC_calendar_date':api.strftime('%Y-%m-%d'),'API_minus_SGML_NY_hours':(api-local).total_seconds()/3600,'API_minus_SGML_fixed_EST_hours':(api-fixed).total_seconds()/3600,'status':'unknown'}
 if not len(available):row['status']='out_of_snapshot';rows.append(row);continue
 next_day=available[0];decision=next_day.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=46);days=(next_day-later_date).days;hours=(decision.tz_convert('UTC')-max(api,local,fixed)).total_seconds()/3600
 row.update(next_Qlib_trade_date=next_day.strftime('%Y-%m-%d'),decision_at_ET=decision.isoformat(),calendar_gap_days=days,minimum_hours_from_latest_clock=hours)
 row['status']='conditional_next_session_clock_pass' if 0<days<=7 and hours>=12 else 'unknown_clock_buffer_or_calendar_gap'
 rows.append(row)
assert len(rows)==64
counts=dict(collections.Counter(x['status'] for x in rows));offsets=dict(collections.Counter(str(x['API_minus_SGML_NY_hours']) for x in rows))
result={'status':'all_64_SEC_6K_acceptance_clock_audited_next_Qlib_session_conditional','plan_sha256':sha(card/'plan.json'),'source_inventory_sha256':src['quarterly_inventory_result_sha256'],'source_NAS_archive_sha256':inventory['NAS_archive_sha256'],'source_NAS_scoped_restore_verified':True,'report_months':len(rows),'status_counts':counts,'API_minus_SGML_NY_hours_counts':offsets,'API_calendar_date_differs_from_SGML':sum(x['API_literal_UTC_calendar_date']!=x['SGML_calendar_date'] for x in rows),'minimum_passing_buffer_hours':min((x['minimum_hours_from_latest_clock'] for x in rows if x['status']=='conditional_next_session_clock_pass'),default=None),'rows':rows,'full_history_first_public_qualified':False,'guidance_semantics_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'conditional_full_SEC_acceptance_date_map_not_factor_ready','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','report_months':len(rows),'status_counts':counts,'API_calendar_date_differences':result['API_calendar_date_differs_from_SGML'],'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],counts,offsets,result['minimum_passing_buffer_hours'])

[97219:MainThread](2026-10-23 05:45:50,053) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/23 05:45:50 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[97219:MainThread](2026-10-23 05:45:50,567) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[97219:MainThread](2026-10-23 05:45:50,567) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


conditional_full_SEC_acceptance_date_map_not_factor_ready {'conditional_next_session_clock_pass': 63, 'out_of_snapshot': 1} {'5.0': 16, '4.0': 48} 20.50611111111111


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_97219/71217719.py:22: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  next_day=available[0];decision=next_day.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=46);days=(next_day-later_date).days;hours=(decision.tz_convert('UTC')-max(api,local,fixed)).total_seconds()/3600


## 中文结论
# TSMC 原始 SEC 6-K：64 期接受时钟与下一交易日

复用 64 个季度月份已识别的原始 6-K（正文原字节和 NAS 限定恢复由上游卡保存），只读原 Stage1 训练价格来源的 Qlib 日历。以 SGML `<ACCEPTANCE-DATETIME>` 日历日期和 SEC submissions API `acceptanceDateTime` 文本的 UTC 日历日期中较晚者为界，取**下一实际 Qlib 交易日 09:46 美东**。同时把 SGML 头分别按纽约当地与固定 EST 解释，并把 API `Z` 字面值按 UTC 解释，要求决策较三个解释中最晚时刻至少 12 小时。

结果：**64 期中 63 期**通过保守下一交易日时钟，最短缓冲 **20.506 小时**；2026-10 事件超出已封存 Qlib 快照，保留 `out_of_snapshot`，不填入最后一天。全部 64 期 API 文本与 SGML 头按纽约时间解释相差恰好 **4 小时（48 期）或 5 小时（16 期）**，但 UTC 与 SGML 的日历日期没有一次不同。不能仅凭尾随 `Z` 把 API 时刻当作无歧义的真实公开时刻；本卡用下一交易日避免日内误用。

首次执行因把上游平铺的 NAS 收据误作嵌套字段，在 Qlib 调用前失败；机械修复只改两个字段名，消费一次 `D.calendar`，0 外部请求、0 市场收益读取、0 拟合、0 账户。SEC [PDS 规范](https://www.sec.gov/files/info/edgar/specifications/pds_dissemination_spec.pdf)区分接受与公开分发，故这些是**条件性日期资格**，不是逐期首次公开页面收据。前瞻营收的币种/会计语义还未统一，也没有 Stage1 信息或 0bp 账户结果；SOXX v2 目标未完成。
